# Interactive K-Means + PCA Explorer

**A.Masmi — Machine Learning Portfolio Project**

This project uses the real **Wine dataset** included with scikit-learn.

### What this notebook does
- Loads and explores the dataset
- Standardizes numerical features
- Tests K-Means from K=2 to K=10
- Creates Elbow and Silhouette graphs
- Selects the best K using Silhouette Score
- Applies PCA
- Shows explained variance
- Creates 2D and 3D interactive cluster visualizations
- Saves the processed data and metadata for a Streamlit live demo


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import joblib

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")


## 1. Load the Real Wine Dataset

In [ ]:
wine = load_wine()

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

df["original_class"] = wine.target
df["original_class_name"] = df["original_class"].map(
    dict(enumerate(wine.target_names))
)

print("Dataset shape:", df.shape)
print("Original classes:", list(wine.target_names))
display(df.head())


## 2. Explore the Dataset

In [ ]:
print("Rows:", len(df))
print("Numerical features:", len(wine.feature_names))
print("\nMissing values:", int(df.isnull().sum().sum()))

print("\nOriginal class distribution:")
display(
    df["original_class_name"]
    .value_counts()
    .rename_axis("Class")
    .to_frame("Samples")
)

print("\nSummary statistics:")
display(df[wine.feature_names].describe().T)


## 3. Standardize the Features

In [ ]:
X = df[wine.feature_names].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original feature matrix:", X.shape)
print("Scaled mean (approximately 0):", round(float(X_scaled.mean()), 6))
print("Scaled standard deviation (approximately 1):", round(float(X_scaled.std()), 6))


## 4. Test K-Means from K=2 to K=10

In [ ]:
k_values = list(range(2, 11))
inertias = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(
        silhouette_score(X_scaled, labels)
    )

k_results = pd.DataFrame({
    "K": k_values,
    "Inertia": inertias,
    "Silhouette_Score": silhouette_scores
})

display(
    k_results.style.format({
        "Inertia": "{:.2f}",
        "Silhouette_Score": "{:.4f}"
    })
)


## 5. Elbow Method

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    k_results["K"],
    k_results["Inertia"],
    marker="o"
)
plt.title("Elbow Method — K-Means")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.grid(alpha=0.25)
plt.show()


## 6. Silhouette Score

In [ ]:
best_k = int(
    k_results.loc[
        k_results["Silhouette_Score"].idxmax(),
        "K"
    ]
)

best_silhouette = float(
    k_results["Silhouette_Score"].max()
)

plt.figure(figsize=(9, 5))
plt.plot(
    k_results["K"],
    k_results["Silhouette_Score"],
    marker="o"
)
plt.axvline(
    best_k,
    linestyle="--",
    label=f"Best K = {best_k}"
)
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(k_values)
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("Best K:", best_k)
print("Best Silhouette Score:", round(best_silhouette, 4))


## 7. Train the Final K-Means Model

In [ ]:
kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans_final.fit_predict(X_scaled)

df["cluster"] = cluster_labels

print("Cluster distribution:")
display(
    df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .to_frame("Samples")
)


## 8. PCA — Dimensionality Reduction

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

variance_df = pd.DataFrame({
    "Principal_Component": [
        f"PC{i+1}" for i in range(len(explained))
    ],
    "Explained_Variance": explained,
    "Cumulative_Variance": cumulative
})

display(
    variance_df.style.format({
        "Explained_Variance": "{:.2%}",
        "Cumulative_Variance": "{:.2%}"
    })
)

pcs_90 = int(np.argmax(cumulative >= 0.90) + 1)

print(
    f"Principal components required to preserve at least 90% variance: {pcs_90}"
)


## 9. PCA Explained Variance Graph

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    variance_df["Principal_Component"],
    variance_df["Explained_Variance"]
)

plt.plot(
    variance_df["Principal_Component"],
    variance_df["Cumulative_Variance"],
    marker="o",
    label="Cumulative variance"
)

plt.axhline(
    0.90,
    linestyle="--",
    label="90% variance"
)

plt.title("PCA Explained Variance")
plt.xlabel("Principal Component")
plt.ylabel("Variance Ratio")
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 10. PCA 2D Projection

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca_2d[:, 0],
    "PC2": X_pca_2d[:, 1],
    "Cluster": cluster_labels.astype(str),
    "Original Class": df["original_class_name"]
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Cluster",
    symbol="Original Class",
    hover_data=["Original Class"],
    title=f"K-Means Clusters in PCA Space — K={best_k}"
)

fig.update_layout(height=650)
fig.show()


## 11. PCA 3D Interactive Visualization

In [ ]:
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

pca_3d_df = pd.DataFrame({
    "PC1": X_pca_3d[:, 0],
    "PC2": X_pca_3d[:, 1],
    "PC3": X_pca_3d[:, 2],
    "Cluster": cluster_labels.astype(str),
    "Original Class": df["original_class_name"]
})

fig3d = px.scatter_3d(
    pca_3d_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Cluster",
    symbol="Original Class",
    hover_data=["Original Class"],
    title=f"Interactive 3D PCA Cluster Explorer — K={best_k}"
)

fig3d.update_layout(height=700)
fig3d.show()


## 12. Compare K-Means Clusters with Original Classes

In [ ]:
comparison = pd.crosstab(
    df["original_class_name"],
    df["cluster"],
    margins=True
)

print("Original class vs K-Means cluster:")
display(comparison)


## 13. Cluster Feature Profiles

In [ ]:
cluster_profiles = (
    df.groupby("cluster")[wine.feature_names]
    .mean()
    .round(2)
)

display(cluster_profiles)


## 14. Save Files for the Streamlit Demo

In [ ]:
# Save dataset with cluster labels
df.to_csv(
    "wine_clustered_data.csv",
    index=False
)

# Save K-selection results
k_results.to_csv(
    "kmeans_scores.csv",
    index=False
)

# Save PCA data
pca_df.to_csv(
    "pca_2d_clusters.csv",
    index=False
)

pca_3d_df.to_csv(
    "pca_3d_clusters.csv",
    index=False
)

# Save cluster profiles
cluster_profiles.to_csv(
    "cluster_profiles.csv"
)

# Save fitted objects
joblib.dump(
    scaler,
    "scaler.joblib"
)

joblib.dump(
    kmeans_final,
    "kmeans_model.joblib"
)

joblib.dump(
    pca_2d,
    "pca_2d.joblib"
)

joblib.dump(
    pca_3d,
    "pca_3d.joblib"
)

metadata = {
    "dataset": "scikit-learn Wine dataset",
    "samples": int(len(df)),
    "features": int(len(wine.feature_names)),
    "best_k": best_k,
    "best_silhouette": best_silhouette,
    "pcs_for_90_percent_variance": pcs_90,
    "feature_names": list(wine.feature_names),
    "target_names": list(wine.target_names)
}

joblib.dump(
    metadata,
    "kmeans_pca_metadata.joblib"
)

print("=" * 60)
print("K-MEANS + PCA PROJECT COMPLETE")
print("=" * 60)

print("Dataset:", metadata["dataset"])
print("Samples:", metadata["samples"])
print("Features:", metadata["features"])
print("Best K:", metadata["best_k"])
print(
    "Best Silhouette Score:",
    round(metadata["best_silhouette"], 4)
)
print(
    "PCs for >=90% variance:",
    metadata["pcs_for_90_percent_variance"]
)

print("\nFiles ready for Streamlit:")
for filename in [
    "wine_clustered_data.csv",
    "kmeans_scores.csv",
    "pca_2d_clusters.csv",
    "pca_3d_clusters.csv",
    "cluster_profiles.csv",
    "scaler.joblib",
    "kmeans_model.joblib",
    "pca_2d.joblib",
    "pca_3d.joblib",
    "kmeans_pca_metadata.joblib"
]:
    print(" ✓", filename)

print("\nREADY FOR STREAMLIT DEPLOYMENT")
